In [ ]:
from pathlib import Path
import os

# Set PROJECT_DATA_DIR before launching Jupyter to use data stored elsewhere.
PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / ".gitignore").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent
DATA_DIR = Path(os.environ.get("PROJECT_DATA_DIR", str(PROJECT_ROOT / "data"))).expanduser()


In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, StratifiedKFold, KFold
from sklearn.metrics import (
    balanced_accuracy_score, roc_auc_score,
    classification_report, confusion_matrix, accuracy_score
)
import seaborn as sns
import matplotlib.pyplot as plt
import shap
from sklearn.linear_model import LogisticRegression
from sklearn.datasets import make_classification
import sklearn
import os


df = pd.read_csv(str(DATA_DIR / 'AI_UNITE_CRSS_FARS_merged_IMPAIRMENT_ONLY.csv'), encoding = 'latin-1')

df = df.dropna(subset=['INJ_SEV'])
df['INJ_SEV'] = df['INJ_SEV'].astype(int)
target = 'INJ_SEV'


categorical_cols = [
    'PERNOTMVIT','PVH_INVL','PERMVIT','MONTH','DAY_WEEK','YEAR',
    'HARM_EV','MAN_COLL','TYP_INT','REL_ROAD','WRK_ZONE','LGT_COND','WEATHER',
    'DRDISTRACT','DRIMPAIR','SPEC_USE','SEX','REST_USE','REST_MIS',
    'HELM_USE','HELM_MIS','DRINKING','ALC_STATUS',
    'ALC_RES','DRUGS','STR_VEH','LOCATION','VE_FORMS','HIT_RUN','BODY_TYP','TOW_VEH',
    'CARGO_BT','HAZ_INV','EMER_USE','DR_PRES','SPEEDREL','VTRAFWAY',
    'VSURCOND','VISION', 'VSPD_LIM', 'VE_TOTAL', 'PEDS' # , 'AIR_BAG','EJECTION','ATST_TYP',
] 
numeric_cols = ['TRAV_SP', 'AGE']

all_columns = categorical_cols + numeric_cols + [target]

X = df[categorical_cols + numeric_cols]
y = df[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)


model = LogisticRegression(multi_class='multinomial', solver='lbfgs', class_weight = 'balanced', max_iter=1000)


model.fit(X_train, y_train)


y_pred = model.predict(X_test)
print(classification_report(y_test, y_pred))
confusion = confusion_matrix(y_test, y_pred)


plt.imshow(confusion, cmap='Blues', interpolation='nearest')
plt.colorbar()
tick_marks = np.arange(2)
thresh = confusion.max() / 2.
for i in range(confusion.shape[0]):
    for j in range(confusion.shape[1]):
        plt.text(j, i, format(confusion[i, j]), ha="center", va="center", color="white" if confusion[i, j] > thresh else "black")

plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.show()


accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy:.2f}\n")


balanced_accuracy = balanced_accuracy_score(y_test, y_pred)
print(f"Balanced Accuracy: {balanced_accuracy:.2f}\n")

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

coef_df = pd.DataFrame(
    model.coef_,
    columns=categorical_cols + numeric_cols,
    index=model.classes_
)

print("Training samples:", X_train.shape[0])
print("Testing samples: ", X_test.shape[0])
print("Sum train + test:", X_train.shape[0] + X_test.shape[0])



In [ ]:
import shap

explainer = shap.LinearExplainer(model, X_train)


sample_size = min(100, len(X_train))
shap_sample = X_train.sample(sample_size, random_state=42)


shap_values = explainer.shap_values(shap_sample)

shap.summary_plot(shap_values, shap_sample, plot_type="bar", max_display=20)
